In [ ]:
# ---------------- Imports ----------------

import json
import os
import re

import numpy as np
import pandas as pd
import yaml


from result_groups import model_set_comparison, k_comparison, label_trajectory_val

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")



In [ ]:
# ---------------- Args ----------------

GROUP_NAME = k_comparison.GROUP_NAME
FILES = k_comparison.FILES


GROUP_NAME = model_set_comparison.GROUP_NAME
FILES = model_set_comparison.FILES

#GROUP_NAME = label_trajectory_val.GROUP_NAME
#FILES = label_trajectory_val.FILES





In [ ]:
# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

with open("../../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

proj_store = config["paths"]["proj-store"]

TRAJECTORY_ROOT = os.path.join(proj_store, "experiments", "ociv-trajectory")

OUTPUT_FOLDER = os.path.join(proj_store, "evaluation", "trajectory-evaluate", "wa-metrics")
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

OUTPUT_PATH = os.path.join(OUTPUT_FOLDER, f"wa-metrics-{GROUP_NAME}.csv")



In [ ]:
# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def read_jsonl(path):
    records = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    return records


def parse_run_filename(path):

    filename = os.path.basename(path)

    match = re.search(r"-(\d+)x(\d+)-k(\d+)-p([\d.]+)-s(\d+)", filename)

    if match is None:
        raise ValueError(f"Could not parse run filename: {filename}")

    return {
        "num_trajectories": int(match.group(1)),
        "trajectory_length": int(match.group(2)),
        "k": int(match.group(3)),
        "p": float(match.group(4)),
        "seed": int(match.group(5)),
    }



def standard_error(values):
    values = pd.Series(values).dropna()

    if len(values) <= 1:
        return np.nan

    return values.std(ddof=1) / np.sqrt(len(values))


def format_mean_se(mean, se, decimals=3):
    if pd.isna(se):
        return f"{mean:.{decimals}f}"

    return f"{mean:.{decimals}f} ± {se:.{decimals}f}"

def get_first(record, keys, required=True):
    for key in keys:
        if key in record:
            return record[key]

    if required:
        raise KeyError(f"None of these keys were found: {keys}")

    return None


def mean_consecutive_run_length(topics):
    topics = list(topics)

    if not topics:
        return np.nan

    run_lengths = []
    current_run_length = 1

    for previous_topic, current_topic in zip(topics[:-1], topics[1:]):
        if current_topic == previous_topic:
            current_run_length += 1
        else:
            run_lengths.append(current_run_length)
            current_run_length = 1

    run_lengths.append(current_run_length)

    return np.mean(run_lengths)



In [ ]:
# ------------------------------------------------------------
# Load trajectory-level metrics
# ------------------------------------------------------------

trajectory_rows = []

for model_name, paths in FILES.items():
    for relative_path in paths:
        full_path = os.path.join(TRAJECTORY_ROOT, relative_path)

        with open(full_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        params = data["params"]
        k = params["candidate_k"]
        seed = params["seed"]

        for trajectory in data["trajectories"]:
            step_rows = []

            for step in trajectory["steps"]:
                candidates = step["candidates"]

                candidate_was = np.array([candidate["human_wa"] for candidate in candidates], dtype=float                )

                selected_wa = float(step["selected_human_wa"])
                candidate_mean_wa = candidate_was.mean()
                candidate_max_wa = candidate_was.max()
                confirmed = bool(step["hypothesis_confirmed"])

                step_rows.append({
                    "selected_wa": selected_wa,
                    "candidate_mean_wa": candidate_mean_wa,
                    "candidate_max_wa": candidate_max_wa,
                    "wa_uplift": selected_wa - candidate_mean_wa,
                    "gap_to_best": candidate_max_wa - selected_wa,
                    "selects_max_wa": float(np.isclose(selected_wa, candidate_max_wa)),
                    "reward": float(confirmed) * selected_wa,
                    "topic_switched": float(step["topic_switched"]),
                    "selected_topic": step["selected_topic"],
                })

            step_df = pd.DataFrame(step_rows)

            trajectory_rows.append({
                "model": model_name,
                "k": k,
                "seed": seed,
                "trajectory_id": trajectory["trajectory_id"],
                "selected_wa": step_df["selected_wa"].mean(),
                "candidate_mean_wa": step_df["candidate_mean_wa"].mean(),
                "candidate_max_wa": step_df["candidate_max_wa"].mean(),
                "wa_uplift": step_df["wa_uplift"].mean(),
                "gap_to_best": step_df["gap_to_best"].mean(),
                "selects_max_wa": step_df["selects_max_wa"].mean(),
                "reward_per_step": step_df["reward"].mean(),
                "total_reward": step_df["reward"].sum(),
                "topic_switch_rate": step_df["topic_switched"].mean(),
                "distinct_topics": step_df["selected_topic"].nunique(),
                "mean_topic_run_length": mean_consecutive_run_length(step_df["selected_topic"]),
            })

trajectory_df = pd.DataFrame(trajectory_rows)

print(trajectory_df.shape)
display(trajectory_df.head())



In [ ]:
# ------------------------------------------------------------
# Seed-level results
# ------------------------------------------------------------

METRICS = [
    "selected_wa",
    "candidate_mean_wa",
    "candidate_max_wa",
    "wa_uplift",
    "gap_to_best",
    "selects_max_wa",
    "reward_per_step",
    "total_reward",
    "topic_switch_rate",
    "distinct_topics",
    "mean_topic_run_length",
]

seed_df = (
    trajectory_df
    .groupby(["model", "k", "seed"], as_index=False)[METRICS]
    .mean()
)

display(seed_df)



In [ ]:
# ------------------------------------------------------------
# Summary across seeds
# ------------------------------------------------------------

summary_rows = []

for (model, k), group in seed_df.groupby(["model", "k"]):
    row = {
        "model": model,
        "k": k,
        "n_seeds": len(group),
    }

    for metric in METRICS:
        row[f"{metric}_mean"] = group[metric].mean()
        row[f"{metric}_se"] = standard_error(group[metric])

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values(["model", "k"]).reset_index(drop=True)

display(summary_df)



In [ ]:
# ------------------------------------------------------------
# Wide k-sensitivity table
# ------------------------------------------------------------

OUTPUT_METRICS = {
    "n_seeds": "n_seeds",
    "selected_wa_mean": "selected_wa",
    "candidate_mean_wa_mean": "candidate_wa",
    "candidate_max_wa_mean": "candidate_max_wa",
    "selects_max_wa_mean": "selects_max_wa",
    "topic_switch_rate_mean": "topic_switch_rate",
    "distinct_topics_mean": "distinct_topics",
}

wide_parts = []

for metric, metric_name in OUTPUT_METRICS.items():
    part = summary_df.pivot(index="k", columns="model", values=metric)
    part = part.reindex(columns=FILES.keys())
    part.columns = [f"{metric_name}_{model}" for model in part.columns]
    wide_parts.append(part)

k_sensitivity_table = pd.concat(wide_parts, axis=1).reset_index()

display(k_sensitivity_table)


k_sensitivity_table.to_csv(OUTPUT_PATH, index=False)



In [ ]:
# ------------------------------------------------------------
# Per-k model comparison tables
# ------------------------------------------------------------

OUTPUT_METRICS = {
    "n_seeds": "n_seeds",
    "selected_wa_mean": "selected_wa",
    "candidate_mean_wa_mean": "candidate_wa",
    "candidate_max_wa_mean": "candidate_max_wa",
    "selects_max_wa_mean": "selects_max_wa",
    "topic_switch_rate_mean": "topic_switch_rate",
    "distinct_topics_mean": "distinct_topics",
    "mean_topic_run_length_mean": "mean_topic_run_length",
}

for k in sorted(summary_df["k"].unique()):
    k_table = summary_df[summary_df["k"] == k].copy()

    k_table = k_table[["model"] + list(OUTPUT_METRICS.keys())]
    k_table = k_table.rename(columns=OUTPUT_METRICS)
    k_table["model"] = pd.Categorical(
        k_table["model"],
        categories=FILES.keys(),
        ordered=True,
    )
    k_table = k_table.sort_values("model").reset_index(drop=True)

    display(f"k: {k}")
    display(k_table)

    k_output_path = os.path.join(
        OUTPUT_FOLDER,
        f"hyperparam-results-{GROUP_NAME}-k{k}.csv",
    )
    k_table.to_csv(k_output_path, index=False)



In [ ]:
# ------------------------------------------------------------
# Standard-error ranges across seeds
# ------------------------------------------------------------

SE_METRICS = [
    "selected_wa",
    "candidate_mean_wa",
    "candidate_max_wa",
    "selects_max_wa",
    "topic_switch_rate",
    "distinct_topics",
    "mean_topic_run_length",
    
]

se_ranges = {}

for metric in SE_METRICS:
    se_col = f"{metric}_se"
    se_ranges[metric] = (summary_df[se_col].min(), summary_df[se_col].max())

for metric, (min_se, max_se) in se_ranges.items():
    print(f"{metric}: SE range = {min_se:.4f}–{max_se:.4f}")
    
    
SE_OUTPUT_PATH = os.path.splitext(OUTPUT_PATH)[0] + "-se-ranges.txt"

with open(SE_OUTPUT_PATH, "w", encoding="utf-8") as f:
    for metric, (min_se, max_se) in se_ranges.items():
        line = f"{metric}: SE range = {min_se:.4f}–{max_se:.4f}"
        print(line)
        f.write(line + "\n")

print(f"Saved SE ranges to {SE_OUTPUT_PATH}")

